In [ ]:
import pandas as pd
import numpy as np
import MDAnalysis as mda
from dask import dataframe as dd

# import prolif as plf

In [ ]:
u = mda.Universe('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/output.top', '/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/5ms-traj.dcd')

In [ ]:
u.trajectory

In [ ]:
test = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/prolif.csv', header = [0,1,2], index_col=0)

In [ ]:
test

In [ ]:

# test[('PRO46')].groupby(axis=1, level=1).sum().sum()
# test.groupby(axis=1, level=0).sum()

In [ ]:
new_header = test.iloc[0] #grab the first row for the header
test = test[1:] #take the data less the header row
test.columns = new_header #set the header row as the df header

In [ ]:
test.columns = ['frame','type','atom_1','atom_2','distance','bad','bad2']

test = test.drop(columns=['bad', 'bad2'])

In [ ]:
size = test.shape[0]

In [ ]:
size

In [ ]:
types = ['']

for type in types:
    test['type'].str.contains(f'sb').value_counts()[True]

In [ ]:
itype_list = ['vdw','sb', 'ps', 'pc', 'ts', 'hb']
itype_dict = {}

for itype in itype_list:
    
    if itype == 'hb':
        try:
            count = test['type'].str.contains('hb').value_counts()[True] + test['type'].str.contains('wb').value_counts()[True]
        except:
            count = 0
    else:
        try:
            count = test['type'].str.contains(f'{itype}').value_counts()[True]
        except:
            count = 0
    
    itype_dict[itype] = count
    
    itype_df = pd.DataFrame(itype_dict, index=[0])

In [ ]:
itype_df

In [ ]:
u = mda.Universe('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/output.pdb', '/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/5ms-traj.dcd')
tm3 = u.select_atoms("protein")
prot = u.select_atoms("protein")
tm3, prot

In [ ]:
# prot-prot interactions
fp = plf.Fingerprint(['Anionic','CationPi','Cationic','EdgeToFace','FaceToFace','HBAcceptor','HBDonor',
 'Hydrophobic','Interaction','MetalAcceptor','MetalDonor','PiCation','PiStacking','VdWContact','XBAcceptor',
 'XBDonor','_BaseCationPi','_BaseHBond','_BaseIonic','_BaseMetallic','_BaseXBond','_Distance'])
fp.run(u.trajectory, tm3, prot)

In [ ]:
df = fp.to_dataframe()

In [ ]:
def rechain_id(read, write):
    with open(read_file, 'r') as old:
        with open(write_file, 'w') as new:
            lines = old.readlines()
            chain = 0
            resid = 1
            for i, line in enumerate(lines):
    #             newline = line
                if i == 0:
                    newline = line
                    new.write(newline)
                elif i == 1:
                    newline = line
                    new.write(newline) 
                elif i == 2:
                    if line.split()[5] == '1':
                        newline = line.replace(' '+line.split()[4]+'   1', ' '+str(chain)+'   1')
                        new.write(newline)
                    if line.split()[5] == '2':
                        newline = line.replace(' '+line.split()[4]+'   2', ' '+str(chain)+'   2')
                        new.write(newline)
                    if line.split()[5] == '3':
                        newline = line.replace(' '+line.split()[4]+'   3', ' '+str(chain)+'   3')
                        new.write(newline)
                else:
                    if len(lines[i].split()) == 5:
                        continue
                    elif len(lines[i].split()) == 12:
                        if lines[i].split()[4] == lines[i-1].split()[4]:
                            if line.split()[5] == '1':
                                newline = line.replace(' '+line.split()[4]+'   1', ' '+str(chain)+'   1')
                                new.write(newline)
                            elif line.split()[5] == '2':
                                newline = line.replace(' '+line.split()[4]+'   2', ' '+str(chain)+'   2')
                                new.write(newline)
                            else:
                                line.split()[5] == '3'
                                newline = line.replace(' '+line.split()[4]+'   3', ' '+str(chain)+'   3')
                                new.write(newline)
                        elif lines[i].split()[4] != lines[i-1].split()[4]:
                            chain += 1
                            if line.split()[5] == '1':
                                newline = line.replace(' '+line.split()[4]+'   1', ' '+str(chain)+'   1')
                                new.write(newline)
                            elif line.split()[5] == '2':
                                newline = line.replace(' '+line.split()[4]+'   2', ' '+str(chain)+'   2')
                                new.write(newline)
                            else:
                                line.split()[5] == '3'
                                newline = line.replace(' '+line.split()[4]+'   3', ' '+str(chain)+'   3')
                                new.write(newline)

In [ ]:
def reres_id(read_file, write_file):
    with open(read_file, 'r') as old:
        with open(write_file, 'w') as new:
            lines = old.readlines()
            resid = 1
            for i, line in enumerate(lines):
                if i == 0:
                    newline = line
                    new.write(newline)
                elif i == 1:
                    newline = line
                    new.write(newline) 
                elif i == 2:
                    newline = line.replace('   '+line.split()[5]+'      ', '   '+str(resid)+'      ')
                    new.write(newline)
                else:
                    if len(lines[i].split()) == 5:
                        continue
                    elif len(lines[i].split()) == 12:
                        if lines[i].split()[5] == lines[i-1].split()[5]:
                            newline = line.replace('   '+line.split()[5]+'      ', '   '+str(resid)+'      ')
                            new.write(newline)
                        elif lines[i].split()[5] != lines[i-1].split()[5]:
                            resid += 1
                            newline = line.replace('   '+line.split()[5]+'      ', '   '+str(resid)+'      ')
                            new.write(newline)

In [ ]:
reres_id(read_file='/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/renum-output-nw.pdb', write_file='/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/final_atmpt.pdb')

In [ ]:
conts = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/res2-contacts.csv')


In [ ]:
new_header = conts.iloc[0] #grab the first row for the header
conts = conts[1:] #take the data less the header row
conts.columns = new_header #set the header row as the df header

In [ ]:
conts

In [ ]:
conts.columns = ['frame','atom_1','atom_2']

In [ ]:
conts = conts.drop(columns=['bad', 'bad2'])

In [ ]:
conts.head(50)

In [ ]:
mask = conts['pc'].str.contains("PHE")

conts = conts.loc[mask]

conts

In [ ]:
longevity = longevity.T
# test.reset_index()
# test.columns = ['Res','Length']

In [ ]:
longevity = longevity.reset_index()

In [ ]:
longevity.columns = ['Res','Length']

In [ ]:
longevity.sort_values(by='Length', ascending=False)

In [ ]:
new_header = test.iloc[0] #grab the first row for the header
test = test[1:] #take the data less the header row
test.columns = new_header #set the header row as the df header

test.columns = ['Frame', 'atom_1', 'atom_2']

mask = test['atom_1'].str.contains(":2") & test['atom_2'].str.contains(":2")

test = test.loc[mask]

# test['atom_1'] = test['atom_1'].str[2:-2]
# test['atom_2'] = test['atom_2'].str[2:-2]

# test = test.drop(columns=['distance', 'bad', 'bad2'])

In [ ]:
mask = test['atom_1'].str.contains("ARG") | test['atom_2'].str.contains("ARG")

In [ ]:
df = test.loc[mask]

In [ ]:
df

In [ ]:
from collections import Counter
from  itertools import combinations

# n = 10
L = Counter([y for x in df.values for y in combinations(x, 2)]).most_common()

pair_df = pd.DataFrame(L, columns=['pair', 'number'])

In [ ]:
pair_df = pair_df.head(100)

In [ ]:
pair_df['number'].mean()

In [ ]:
df.duplicated(subset=['atom_1', 'atom_2']).sum()

In [ ]:
value

In [ ]:
res = 'PHE'
sizes = [test[f'{res}'][0]/10, test[f'{res}'][1], test[f'{res}'][2], test[f'{res}'][3],  test[f'{res}'][4], test[f'{res}'][5]]
names = ['VDW','Salt-Bridge','Pi-Stacking','Pi-Cation', 'T-stacking','H-Bonding']
fig, ax = plt.subplots()
wedges, texts = ax.pie(sizes);
ax.legend(wedges, names)

In [ ]:
try:
    value = df['type'].str.contains(f'vdw').value_counts()[True]
    print(value)
except:
    value = 0

In [ ]:
df

In [ ]:
itype_list = ['vdw','sb', 'ps', 'pc', 'ts', 'hb']
itype_dict = {}

for itype in itype_list:
    
    if itype == 'hb':
        try:
            count = df['type'].str.contains('hb').value_counts()[True] + df['type'].str.contains('wb').value_counts()[True]
        except:
            count = 0
    else:
        try:
            count = df['type'].str.contains(f'{itype}').value_counts()[True]
        except:
            count = 0
    
    itype_dict[itype] = count
    
    itype_df = pd.DataFrame(itype_dict, index=[0])
    
        
        

In [ ]:
itype_list = ['vdw','sb', 'ps', 'pc', 'ts', 'hb']
loop = 0
final_df = pd.DataFrame(index=itype_list)

In [ ]:
final_df

In [ ]:
final_df = final_df.reset_index()
final_df.columns = ['Type']

In [ ]:
# test = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/all-contacts.csv')

# new_header = test.iloc[0] #grab the first row for the header
# test = test[1:] #take the data less the header row
# test.columns = new_header #set the header row as the df header

# test.columns = ['Frame', 'type', 'atom_1', 'atom_2', 'distance', 'bad', 'bad2']

# test = test.drop(columns=['distance', 'bad', 'bad2'])

res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
            'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

itype_list = ['vdw','sb', 'ps', 'pc', 'ts', 'hb','hp']
loop = 0
final_df = pd.DataFrame(index=itype_list)
final_df = final_df.reset_index()
final_df.columns = ['Type']

for res in res_list:
    
    itype_dict = {}
    
    mask = test['atom_1'].str.contains(f"{res}") | test['atom_2'].str.contains(f"{res}")
    df = test.loc[mask]
    
    for itype in itype_list:
    
        if itype == 'hb':
            try:
                count = df['type'].str.contains('hb').value_counts()[True] + df['type'].str.contains('lb').value_counts()[True]
            except:
                count = 0
        else:
            try:
                count = df['type'].str.contains(f'{itype}').value_counts()[True]
            except:
                count = 0

        itype_dict[itype] = count
        loop += 1
        
    final_df[f'{res}'] = final_df['Type'].map(itype_dict)
    
final_df.to_csv(f'type_freqs.csv')
    

In [ ]:
okay = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/type_freqs.csv')

In [ ]:
test['type'].unique()

In [ ]:
okay

In [ ]:
okay = okay.T
new_header = okay.iloc[0] #grab the first row for the header
okay = okay[1:] #take the data less the header row
okay.columns = new_header #set the header row as the df header
okay = okay.reset_index()

In [ ]:
okay

In [ ]:
okay.columns = ['Type','ALA']

In [ ]:
okay = okay.reset_index()

In [ ]:
test

In [ ]:
test['atom_1'].str.contains(f'HIS:2').value_counts()[True]

In [ ]:
new_header = test.iloc[0] #grab the first row for the header
test = test[1:] #take the data less the header row
test.columns = new_header #set the header row as the df header

In [ ]:
test.columns = ['Frame', 'type', 'atom_1', 'atom_2', 'distance', 'bad', 'bad2']

In [ ]:
test = test.drop(columns=['distance', 'bad', 'bad2'])

In [ ]:
blah = test.values

In [ ]:
df = pd.DataFrame(blah, columns=['frame','type','atom_1','atom_2'])

In [ ]:
test = test[~test.type.str.contains("vdw")]

In [ ]:
test_list = test.to_numpy().tolist()

In [ ]:
type(test_list)

In [ ]:
r1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-1/cont_freqs.csv')
r2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-2/cont_freqs.csv')
r3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-3/cont_freqs.csv')

r4 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/cont_freqs.csv')
r5 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-2/cont_freqs.csv')
r6 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-3/cont_freqs.csv')

r7 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-1/cont_freqs.csv')
r8 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-2/cont_freqs.csv')
r9 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-3/cont_freqs.csv')

r1 = r1.T
r2 = r2.T
r3 = r3.T

r4 = r4.T
r5 = r5.T
r6 = r6.T

r7 = r7.T
r8 = r8.T
r9 = r9.T

r1.columns = ['Average Contacts/Frame']
r2.columns = ['Average Contacts/Frame']
r3.columns = ['Average Contacts/Frame']

r4.columns = ['Average Contacts/Frame']
r5.columns = ['Average Contacts/Frame']
r6.columns = ['Average Contacts/Frame']

r7.columns = ['Average Contacts/Frame']
r8.columns = ['Average Contacts/Frame']
r9.columns = ['Average Contacts/Frame']

r1 = r1.sort_values(by='Average Contacts/Frame')
r2 = r2.sort_values(by='Average Contacts/Frame')
r3 = r3.sort_values(by='Average Contacts/Frame')

r4 = r4.sort_values(by='Average Contacts/Frame')
r5 = r5.sort_values(by='Average Contacts/Frame')
r6 = r6.sort_values(by='Average Contacts/Frame')

r7 = r7.sort_values(by='Average Contacts/Frame')
r8 = r8.sort_values(by='Average Contacts/Frame')
r9 = r9.sort_values(by='Average Contacts/Frame')

r1.plot(kind='bar', title='50-1')
r2.plot(kind='bar', title='50-2')
r3.plot(kind='bar', title='50-3')

r4.plot(kind='bar', title='125-1')
r5.plot(kind='bar', title='125-2')
r6.plot(kind='bar', title='125-3')

r7.plot(kind='bar', title='250-1')
r8.plot(kind='bar', title='250-2')
r9.plot(kind='bar', title='250-3')

r10 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/cont_freqs2.csv')
r10 = r10.T
r10.columns = ['Average Contacts/Frame']
r10 = r10.sort_values(by='Average Contacts/Frame')
r10.plot(kind='bar', title='125-1 NO VDW')

In [ ]:
# Python program to convert .tsv file to .csv file
# importing re library
import re
  
# reading given tsv file
with open('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/res-contacts_nobb.tsv', 'r') as myfile:  
    with open('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/res-contacts_nobb.csv', 'w') as csv_file:
        for line in myfile:
        
            # Replace every tab with comma
            fileContent = re.sub("\t", ",", line)

            # Writing into csv file
            csv_file.write(fileContent)
  
# output
print("Successfully made csv file")